In [17]:
import torch
from importlib.metadata import version
import torch.nn.functional as F
import torch.nn as nn
print("torch version:", version("torch"))

torch version: 2.5.1


# non trainable version

In [4]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [12]:
attention_scores = F.softmax(inputs @ inputs.T)

/tmp/ipykernel_307748/1352524120.py:1: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  attention_scores = F.softmax(inputs @ inputs.T)


In [14]:
all_context_vecs = attention_scores @ inputs

In [16]:
all_context_vecs

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

# 3.4.2 compact SelfAttention class

In [21]:
class SelfAttention(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        
    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        attention_scores = queries @ keys.T
        attention_weights = F.softmax(attention_scores, dim=-1)
        
        context_vec = attention_weights @ values
        return context_vec

In [24]:
torch.manual_seed(789)
d_in = 3
d_out = 2
sa_v2 = SelfAttention(d_in, d_out)
print(sa_v2(inputs).data)

tensor([[-0.0726,  0.0731],
        [-0.0740,  0.0716],
        [-0.0740,  0.0715],
        [-0.0756,  0.0692],
        [-0.0760,  0.0685],
        [-0.0748,  0.0703]])


# 3.5 Causal attention

In [32]:
context_length = attention_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length), diagonal=0)
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [34]:
attention_scores*mask_simple

tensor([[0.2098, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1385, 0.2379, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1390, 0.2369, 0.2326, 0.0000, 0.0000, 0.0000],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.0000, 0.0000],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.0000],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [46]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attention_scores.masked_fill(mask.bool(), -torch.inf)
masked

tensor([[0.2098,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.1385, 0.2379,   -inf,   -inf,   -inf,   -inf],
        [0.1390, 0.2369, 0.2326,   -inf,   -inf,   -inf],
        [0.1435, 0.2074, 0.2046, 0.1462,   -inf,   -inf],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879,   -inf],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

## with dropout

In [49]:
torch.manual_seed(123)
dropout = nn.Dropout(p=0.5)
example = torch.ones(6,6)
print(dropout(example))
print(dropout(attention_scores))

tensor([[2., 2., 2., 2., 2., 2.],
        [0., 2., 0., 0., 0., 0.],
        [0., 0., 2., 0., 2., 0.],
        [2., 2., 0., 0., 0., 2.],
        [2., 0., 0., 0., 0., 2.],
        [0., 2., 0., 0., 0., 0.]])
tensor([[0.4197, 0.4012, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.4665, 0.0000, 0.0000, 0.3162],
        [0.0000, 0.0000, 0.4652, 0.2484, 0.0000, 0.3129],
        [0.0000, 0.4148, 0.0000, 0.2924, 0.2526, 0.0000],
        [0.0000, 0.3917, 0.3950, 0.2734, 0.3757, 0.2590],
        [0.2769, 0.4367, 0.0000, 0.0000, 0.1976, 0.3791]])


In [54]:
batch = torch.stack([inputs, inputs], dim=0)
print(batch.shape)

torch.Size([2, 6, 3])


In [65]:
batch.shape

torch.Size([2, 6, 3])

In [66]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False, dropout=0.0, context_length=6):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        print(x.shape)
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attention_scores = queries @ keys.transpose(1, 2)
        attention_scores.masked_fill_(self.mask.bool()[:num_tokens,:num_tokens], -torch.inf)

        attention_weights = F.softmax(masked, dim=-1)

        context_vec = attention_weights @ values
        return context_vec

In [68]:
torch.manual_seed(123)

context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length=context_length, dropout=0.0)
context_vecs = ca(batch)
print(context_vecs)
print(context_vecs.shape)

torch.Size([2, 6, 3])
tensor([[[-0.4519,  0.2216],
         [-0.5896,  0.0024],
         [-0.6317, -0.0659],
         [-0.5695, -0.0854],
         [-0.5521, -0.0994],
         [-0.5320, -0.1086]],

        [[-0.4519,  0.2216],
         [-0.5896,  0.0024],
         [-0.6317, -0.0659],
         [-0.5695, -0.0854],
         [-0.5521, -0.0994],
         [-0.5320, -0.1086]]], grad_fn=<UnsafeViewBackward0>)
torch.Size([2, 6, 2])


# Multi-head attention

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length=6, dropout=0.0, num_heads=2, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias) # d_in*d_out trainable parameters
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias) # d_in*d_out trainable parameters
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias) # d_in*d_out trainable parameters
        self.out_proj = nn.Linear(d_out, d_out) # d_out*d_out + d_out(bias) trainable parameters
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )
        
    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        
        attention_scores = queries @ keys.transpose(2,3)
        mask_bool = self.mask.bool()[:num_tokens,:num_tokens]
        attention_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = F.softmax(attention_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        context_vec = (attn_weights @ values).transpose(1, 2).contiguous()
        context_vec = context_vec.view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)
        return context_vec
        

In [82]:
torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape
d_out = 2

mha = MultiHeadAttention(d_in, d_out, context_length=context_length, dropout=0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print(context_vecs.shape)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
torch.Size([2, 6, 2])


In [91]:
sum(p.numel() for p in mha.parameters() if p.requires_grad)

24

In [92]:
batch.shape

torch.Size([2, 6, 3])